# Workshop Demo: ChemProp and ChemTorch

Welcome to this workshop on using ChemTorch, a modular python package for machine learning for reaction properties, as well as ChemProp, a python package for molecular machine learning.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import numpy as np
import chemprop

from sklearn.decomposition import PCA

import os
from copy import deepcopy
from functools import partial

import torch
from lightning import Trainer, seed_everything
from torch_geometric.loader import DataLoader
from torchmetrics import MeanAbsoluteError, MeanSquaredError, MetricCollection

from chemtorch.components.data_pipeline import SimpleDataPipeline
from chemtorch.components.data_pipeline.column_mapper import ColumnFilterAndRename
from chemtorch.components.data_pipeline.data_source import SingleCSVSource
from chemtorch.components.data_pipeline.data_splitter import RatioSplitter

from chemtorch.components.representation.graph import CGR
from chemtorch.components.representation.graph.featurizer import (
    AtomDegreeFeaturizer,
    AtomFormalChargeFeaturizer,
    AtomHCountFeaturizer,
    AtomHybridizationFeaturizer,
    AtomIsAromaticFeaturizer,
    AtomIsInRingFeaturizer,
    BondInRingFeaturizer,
    BondIsConjugatedFeaturizer,
    BondTypeFeaturizer,
    CentiAtomMassFeaturizer,
    FeaturizerCompose,
    OrganicAtomicNumberOneHotFeaturizer,
)

from chemtorch.components.layer import LayerStack
from chemtorch.components.layer.gnn_layer import (
    DMPNNBlock,
    DMPNNConv,
    DMPNNStack,
    EdgeToNodeEmbedding,
)


from chemtorch.components.model import MLP
from chemtorch.components.model.gnn import (
    DirectedEdgeEncoder,
    GNN,
    GlobalPool,
)

from chemtorch.core import DataModule
from chemtorch.core.routine import RegressionRoutine
from chemtorch.core.scheduler import CosineWithWarmupLR
from chemtorch.utils import Standardizer

# Part A: Inspecting the data

For this workshop, we will work with an experimental database of barrier heights of S$_N$Ar reactions [1]. We will download a preprocessed version:

In [ ]:
!wget https://raw.githubusercontent.com/hesther/teaching/refs/heads/main/camlc2026/snar_reactions.csv

In [ ]:
df = pd.read_csv("snar_reactions.csv")
df

We see that the dataset consists of 442 reactions (specified as atom-mapped reaction SMILES without explicit hydrogens). Let us plot the histogram of labels:

In [ ]:
plt.hist(df['exp_activation_energy'])

Finally, as a baseline, we will compute the performance of a model predicting only the mean of the label distribution:

In [ ]:
mean_absolute_error(df['exp_activation_energy'], [np.mean(df['exp_activation_energy'])]*len(df))

Any machine learning model we build should be significantly better than a mean absolute error of 2.86kcal/mol!

### Exercise:
Compute the root mean squared error (using the function `root_mean_squared_error()`) of a model predicting only the mean of the label distribution, similar to the mean absolute error above!

# Part B: Chemprop

Chemprop [2] is a machine learning architecture for molecular (mainly) and reaction (additionally) property prediction based on the D-MPNN architecture, which comes with many convenience functionality like uncertainty prediction and more (checkout the documentation [here](https://chemprop.readthedocs.io/en/main/), which comes with many examples and tutorials). In the following, we will explore Chemprop's default capabilities to fit our S$_N$Ar data with

1. Molecular encodings, focusing on the reactants only
2. Reaction encodings, taking into account reactants and products.

## Part B-1: Molecules and Chemprop

Since our dataset above consists of whole reactions, we will need to preprocess it to make a column for reactants only:



In [ ]:
df['reactants'] = [s.split(">")[0] for s in df['AAM']]
df[['reactants','exp_activation_energy']].to_csv('snar_reactions_reactants.csv', index=False)

We can run chemprop either from the command line, or as python code. Let us try the CLI version first:

In [ ]:
! chemprop train --data-path snar_reactions_reactants.csv  --task-type regression --metrics mae rmse

We get a model with mae=1.66kcal/mol and rmse=2.18kcal/mol. Definitely better than our mean predictor, but not great yet. This is because we are currently not using the product information at all. However, it comes with the benefit that most of Chemprop's functionality is centered around molecular properties. For example, there are several pretrained models available, which we can download and finetune. For example, let's give CheMeleon a try (this fill preload the encoding = message passing part of the architecture, and only initialize the parameters of the NN readout from scratch, so we should need less epochs to train), However, the CheMeleon foundation model is very big (9M parameters instead of the 200k above), so the training is rather slow, and we will only do 3 epochs here:

In [ ]:
! chemprop train --data-path snar_reactions_reactants.csv  --task-type regression --metrics mae rmse --from-foundation CheMeleon --epochs 3

### Exercise:
Read the section of the Chemprop documentation on adding 2D fingerprints as additional input [here](https://chemprop.readthedocs.io/en/main/tutorial/cli/train.html#molecule-level-2d-features) and try one of them specifying `--molecule-featurizers` (copy/paste the command without the foundation model and add this keyword).

## Part B-2: Reactions and Chemprop
Chemprop can also encode reactions, as condensed graph of reaction. For this, we can simply specify columns as holding reaction SMILES instead of molecular SMILES using the `--reaction-column option` (and switch to the CSV file with the atom-mapped reaction SMILES):

In [ ]:
! chemprop train --data-path snar_reactions.csv  --task-type regression --metrics mae rmse --reaction-columns AAM

With using reaction mode, we can thus get quite a performance boost (mae=1.40kcal/mol and rmse=1.79kcal/mol versus in molecular mode mae=1.66kcal/mol and rmse=2.18kcal/mol). However, some features in Chemprop are not available within reaction mode, and the architecture and representation are limited to D-MPNN and CGR, respectively.

Key take-aways:
- Chemprop is great for quick benchmarking, especially for molecular datasets
- Chemprop also works for reaction datasets, albeit with a bit less functionality

Note: Chemprop has over 20 Jupyter notebook tutorials in their documentation, make sure to check them out [here](https://chemprop.readthedocs.io/en/main/notebooks.html). You can also use Chemprop or any of its parts within Python code. For example, copy-pasting together parts of the Chemprop tutorial on visualizing encodings, as well as the tutorial on reaction datasets (you will need to update the checkpoint path!)

In [ ]:
mpnn = chemprop.models.MPNN.load_from_checkpoint('/content/chemtorch/chemprop_training/snar_reactions/2026-05-18T20-09-50/model_0/checkpoints/best-epoch=49-val_loss=0.17.ckpt')
mpnn.eval()

In [ ]:
featurizer = chemprop.featurizers.CondensedGraphOfReactionFeaturizer(mode_="REAC_DIFF")
all_data = [chemprop.data.ReactionDatapoint.from_smi(smi) for smi in df['AAM']]
dset = chemprop.data.ReactionDataset(all_data, featurizer=featurizer)
all_loader = chemprop.data.build_dataloader(dset, shuffle=False)


with torch.no_grad():
    encodings = [
        mpnn.encoding(batch.bmg, batch.V_d, batch.X_d, i=1)
        for batch in all_loader
    ]
    encodings = torch.cat(encodings, 0)

print(encodings.shape)

encodings = encodings.detach()
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(encodings)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(1, 1, 1)
ax.set_title("Encodings")
ax.set_xlabel('PCA1'); ax.set_ylabel('PCA2')

ax.scatter(principalComponents[:, 0], principalComponents[:, 1], c=df['exp_activation_energy'])
plt.show()

# Part C: ChemTorch

One of the main limitations of Chemprop is it's limitation on only the D-MPNN architecture, as well as its limitations on reaction representations and functionality.

We will therefore take a look at ChemTorch, a python package specifically designed for ML on reaction datasets.

First, we will train again a  (D-MPNN) for comparability.

## Part C-1: ChemTorch on the commandline

You can easily train a ChemTorch model, for example, with the preset `+experiment=graph` we obtain a D-MPNN model, and simply have to specificy the data path and column identifiers:

In [ ]:
!chemtorch +experiment=graph data_module.data_pipeline.data_source.data_path=snar_reactions.csv data_module.data_pipeline.column_mapper.smiles='AAM' data_module.data_pipeline.column_mapper.label='exp_activation_energy' log=false trainer.max_epochs=50

After a minute or so, you have your very first trained ChemTorch model! For a lot of tasks, this would already be sufficient, and you can check out the documentation [here](https://heid-lab.github.io/chemtorch) for further options. For example, if you would like a GCN instead of the D-MPNN, you simply specify `model=gcn` additionally (other graph neural network options include GAT, GATv2, GatedGCN, and GINE).

### Exercise:
Choose another architecture option apart from a D-MPNN and train it. Compare the model architecture, number of parameters and performance.

### Exercise:
Seems like the D-MPNN training above has not quite converged. Try increasing the epochs (e.g. to 100 or 200) and monitor the performance. Optionally also increase the number of epochs for our Chemprop training, for fair comparison.

## Part C-2: ChemTorch as Python code

Now, for let us take a deeper dive into the ChemTorch pipeline and make our own from scratch:

In [ ]:
import os
from copy import deepcopy
from functools import partial

import torch
from lightning import Trainer, seed_everything
from torch_geometric.loader import DataLoader
from torchmetrics import MeanAbsoluteError, MeanSquaredError, MetricCollection

from chemtorch.components.data_pipeline import SimpleDataPipeline
from chemtorch.components.data_pipeline.column_mapper import ColumnFilterAndRename
from chemtorch.components.data_pipeline.data_source import SingleCSVSource
from chemtorch.components.data_pipeline.data_splitter import RatioSplitter

from chemtorch.components.representation.graph import CGR
from chemtorch.components.representation.graph.featurizer import (
    AtomDegreeFeaturizer,
    AtomFormalChargeFeaturizer,
    AtomHCountFeaturizer,
    AtomHybridizationFeaturizer,
    AtomIsAromaticFeaturizer,
    AtomIsInRingFeaturizer,
    BondInRingFeaturizer,
    BondIsConjugatedFeaturizer,
    BondTypeFeaturizer,
    CentiAtomMassFeaturizer,
    FeaturizerCompose,
    OrganicAtomicNumberOneHotFeaturizer,
)

from chemtorch.components.layer import LayerStack
from chemtorch.components.layer.gnn_layer import (
    DMPNNBlock,
    DMPNNConv,
    DMPNNStack,
    EdgeToNodeEmbedding,
)


from chemtorch.components.model import MLP
from chemtorch.components.model.gnn import (
    DirectedEdgeEncoder,
    GNN,
    GlobalPool,
)

from chemtorch.core import DataModule
from chemtorch.core.routine import RegressionRoutine
from chemtorch.core.scheduler import CosineWithWarmupLR
from chemtorch.utils import Standardizer

In [ ]:
seed = 0
seed_everything(seed)
torch.use_deterministic_algorithms(True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = (":4096:8")     # https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility

## 1. Data Pipeline

In ChemTorch, the data pipeline is responsible for loading the raw data and turning into a format that can be consumed by downstream code.

Here is what the `SimpleDataPipeline` does:

- it loads the raw data from a data source, e.g. a CSV file,
- selects/renames the data columns to align with the ChemTorch API,
- splits the data into train/validation/test sets.


In [ ]:
DATA_PATH = "snar_reactions.csv"

pipeline = SimpleDataPipeline(
    data_source=SingleCSVSource(data_path=DATA_PATH),
    column_mapper=ColumnFilterAndRename(smiles="AAM", label="exp_activation_energy"),
    data_splitter=RatioSplitter(train_ratio=0.8, val_ratio=0.1, test_ratio=0.1),
)

data_split = pipeline()
data_split.train.head()

## 2. Reaction Representation

One of the most important questions in chemical deep learning is how to represent your molecules/reactions for your desired task. We will be using the Condensed Graph of Reaction (CGR) [3] here. It it a graph overlay of reactant and product graphs where forming/breaking bonds carry special annotations.

As shown in the ChemTorch paper [4], the CGR provides a strong baseline for barrier height prediction but it requires atom mapping.

In ChemTorch, each representation takes in a SMILES string and returns some kind of data object compatible with your model. The CGR representation is initilized with atom and bond featurizers which can be composed. The following featurizers are simple RDKit featurizers preconfigured for organic molecules.

In [ ]:
atom_featurizer = FeaturizerCompose(
    [
        OrganicAtomicNumberOneHotFeaturizer(),
        AtomDegreeFeaturizer(),
        AtomFormalChargeFeaturizer(),
        AtomHCountFeaturizer(),
        AtomHybridizationFeaturizer(),
        AtomIsAromaticFeaturizer(),
        CentiAtomMassFeaturizer(),
        AtomIsInRingFeaturizer(),
    ]
)

bond_featurizer = FeaturizerCompose(
    [
        BondTypeFeaturizer(),
        BondIsConjugatedFeaturizer(),
        BondInRingFeaturizer(),
    ]
)

representation = CGR(
    atom_featurizer=atom_featurizer,
    bond_featurizer=bond_featurizer,
)

In [ ]:
example_smiles = data_split.train.iloc[0]["smiles"]
example_graph = representation.construct(example_smiles)

print(example_graph)
print("node feature dimension:", example_graph.x.shape[-1])
print("edge feature dimension:", example_graph.edge_attr.shape[-1])

Our featurization results in 88 node features and 22 edge features.

### Exercise (no code):
For any SMILES string of your choice, e.g. ethane `CC`, try to draw out the molecular structure, and think about some of the atomic properties above. E.g., what will the `AtomHCountFeaturizer` yield, or the `AtomIsAromaticFeaturizer`?



## 3. DataModule

One of the heart pieces in the ChemTorch pipeline is the `DataModule` which extends PyTorch Lightning’s `lightning.DataModule`. The data module manages data object creation and batching for training/inference.

It is initialized with

- our data pipeline,
- the chosen representation,
- a compatible torch dataloader class, and
- optional data transforms/augmentations.

For graph data we use PyG’s DataLoader, partially configured with the batch size. The data module will use this as a template to create train, validation, and test dataloaders.

In [ ]:
dataloader_factory = partial(
    DataLoader,
    batch_size=64,
    num_workers=0,
    pin_memory=False,
)

data_module = DataModule(
    data_pipeline=pipeline,
    representation=representation,
    dataloader_factory=dataloader_factory,
    transform=None,
    augmentations=None
)

In [ ]:
batch, labels = next(iter(data_module.train_dataloader()))
print(batch)
print(labels.shape)

## 4. D-MPNN Model

ChemTorch supports any type of model compatbile with the PyTorch API.

For GNNs, ChemTorch comes with a ready to use blueprint that is initialized with the following components:

1. `encoder`: turns the node/edge features into learned embeddings.
2. `layer_stack`: applies repeated graph convolutions, in our case, a few blocks of directed message passing.
To increase the expressivity of our GNN, each block adds a residual connection, a 2-layer feedforward network, and dropout.
These are applied after message passing and before ReLU.

3. `pool`: reduces node/edge embeddings to one graph embedding per reaction.
4. `head`: maps the graph embeddings to the regression prediction, in our case a scalar barrier height.

In [ ]:
NUM_NODE_FEATURES = 88
NUM_EDGE_FEATURES = 22
DIRECTED_EDGE_ENCODER_IN_CHANNELS = NUM_NODE_FEATURES + NUM_EDGE_FEATURES

HIDDEN_CHANNELS = 128
DEPTH = 3
OUT_CHANNELS = 1

# 1. directed edge encoder
encoder=DirectedEdgeEncoder(
    in_channels=DIRECTED_EDGE_ENCODER_IN_CHANNELS,
    out_channels=HIDDEN_CHANNELS,
)

# 2. directed message passing layer stack
layer_block = DMPNNBlock(
    graph_conv=DMPNNConv(
        in_channels=HIDDEN_CHANNELS,
        out_channels=HIDDEN_CHANNELS,
    ),
    residual=True,
    ffn=True,
    dropout=0.1,
    act="relu",
    hidden_channels=HIDDEN_CHANNELS
)

layer_stack = DMPNNStack(
    dmpnn_blocks=LayerStack(
        layer=layer_block,
        depth=DEPTH,
    ),
    edge_to_node_embedding=EdgeToNodeEmbedding(
        embedding_size=HIDDEN_CHANNELS,
        num_node_features=NUM_NODE_FEATURES,
    ),
)

# 3. pooling function
pool = GlobalPool(aggr="add")

# 4. prediction head
head = MLP(
        in_channels=HIDDEN_CHANNELS,
        hidden_size=HIDDEN_CHANNELS,
        num_hidden_layers=1,
        out_channels=OUT_CHANNELS,
        dropout=0.01,
        act="relu",
    )

# assmeble everything into the GNN
model = GNN(
    encoder=encoder,
    layer_stack=layer_stack,
    pool=pool,
    head=head,
)

print(model)

We can now use our model to make predictions (which are not great because we didn't train yet):

In [ ]:
with torch.no_grad():
    predictions = model(deepcopy(batch))

print(predictions.shape)
print(predictions[:10])

## 5. Routine

To reduce boilerplate PyTorch code, ChemTorch builds on PyTorch Lightning’s `LightningModule` and provides a common ``Routine abstraction that handles training and inference logic such as metric logging, optimization steps, learning-rate scheduling, and prediction rescaling for regression tasks.

The `Routine` wraps our PyTorch model together with its loss function, optimizer, learning-rate scheduler, metrics, and prediction postprocessing.

The `Standardizer` normalizes targets using statistics computed from the training set and later reverses this transformation to compute metrics and return predictions in the original scale.

In [ ]:
# train data statistics from previous experiment
TRAIN_LABEL_MEAN = 21.4
TRAIN_LABEL_STD = 3.9

metrics = {
    "train": MetricCollection(
        {"rmse": MeanSquaredError(squared=False)}
    ),
    "val": MetricCollection(
        {"rmse": MeanSquaredError(squared=False)}
    ),
    "test": MetricCollection(
        {
            "mae": MeanAbsoluteError(),
            "rmse": MeanSquaredError(squared=False),
        }
    ),
}

routine = RegressionRoutine(
    model=model,
    standardizer=Standardizer(mean=TRAIN_LABEL_MEAN, std=TRAIN_LABEL_STD),
    loss=torch.nn.MSELoss(),
    optimizer=partial(
        torch.optim.AdamW,
        lr=0.00053,
        weight_decay=0.0006
    ),
    lr_scheduler={
        "scheduler": partial(
            CosineWithWarmupLR,
            num_warmup_steps=10,
            num_training_steps=100,
            start_factor=1.0e-6,
            end_factor=1.0,
            eta_min=0.0,
        ),
        "interval": "epoch",
        "frequency": 1,
    },
    metrics=metrics,
)

## 6. Trainer

Finally, Lightning’s `Trainer` orchestrates fitting and testing.

For the tutorial we use a maximum of 50 epoch, no logger, and disable checkpointing.

When running a real experiment, we would use a higher maximum number of epochs, the Weights & Biases Logger, and enable checkpointing.

In [ ]:
MAX_EPOCHS = 50

trainer = Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    logger=False,
    enable_checkpointing=False,
    gradient_clip_val=1.0,
    log_every_n_steps=1,
)

In [ ]:
trainer.fit(routine, datamodule=data_module)

In [ ]:
trainer.test(routine, datamodule=data_module)

In [ ]:
plt.scatter(data_split.test['label'],trainer.predict(routine, data_module.test_dataloader())[0])
plt.xlabel("Experimental activation energy")
plt.xlabel("Predicted activation energy")

Now, you can make any changes to the model, e.g. change the atom/bond featurizer, change the architecture, change individual layers, aggregation, etc. and design the model of your dreams. This is the power of building your own pipeline: You can mix and match any components you like, or implement any custom components.

For example, let us build our own additional atom feature if we expect radicals to be in our dataset:

In [ ]:
from rdkit.Chem import Atom

from chemtorch.components.representation.graph.featurizer.featurizer_base import FeaturizerBase


class AtomGetNumRadicalFeaturizer(FeaturizerBase[Atom]):
    """Atom number of radical electrons featurizer."""

    def __init__(self):
        features = [Atom.GetNumRadicalElectrons]
        super().__init__(features)



In [ ]:
atom_featurizer = FeaturizerCompose(
    [
        OrganicAtomicNumberOneHotFeaturizer(),
        AtomDegreeFeaturizer(),
        AtomFormalChargeFeaturizer(),
        AtomHCountFeaturizer(),
        AtomHybridizationFeaturizer(),
        AtomIsAromaticFeaturizer(),
        CentiAtomMassFeaturizer(),
        AtomIsInRingFeaturizer(),
        AtomGetNumRadicalFeaturizer(),

    ]
)

representation = CGR(
    atom_featurizer=atom_featurizer,
    bond_featurizer=bond_featurizer,
)

In [ ]:
example_smiles = "[CH2:1]>>[CH4:1]"
example_graph = representation.construct(example_smiles)
print(example_graph.x.shape)
example_graph.x[0][:45]

### Exercise:
Change the model architecture or featurization, for example, what happens if you only use the Atomic Number Featurizer and delete the rest?

Note that ChemTorch also supports other modalities than graph neural networks on graphs: Neural networks on fingerprints, language models on tokens (string representations), or spatial neural networks on 3D coordinates. Consult the documentation or the manuscript for examples!

# References

[1] Jorner, K.; Brinck, T.; Norrby, P.-O.; Buttar, D. Machine learning meets mechanistic modelling for accurate prediction of experimental activation energies. Chem. Sci. 2021, 12, 1163-1175, DOI: 10.1039/D0SC04896H

[2]
- v0: Yang, K.; Swanson, K.; Jin, W.; Coley, C.; Eiden, P.; Gao, H.; Guzman-Perez, A.; Hopper, T.; Kelley, B.; Mathea, M. Analyzing learned molecular representations for property prediction. J. Chem. Inf. Model. 2019, 59, 3370– 3388, DOI: 10.1021/acs.jcim.9b00237

- v1: Heid, E.; Greenman, K. P.; Chung, Y.; Li, S.-C.; Graff, D. E.; Vermeire, F. H.; Wu, H; Green, W. H.; McGill, C. J. Chemprop: a machine learning package for chemical property prediction. J. Chem. Inf. Model. 2024, 64, 9–17 DOI: 10.1021/acs.jcim.3c01250

[3] Heid, E.; Green, W. H. Machine learning of reaction properties via learned representations of the condensed graph of reaction. J. Chem. Inf. Model. 2022, 62, 2101– 2110, DOI: 10.1021/acs.jcim.1c00975

[4] De Landsheere, J.; Zamyatin, A.; Karwounopoulo, J.; Heid, E. ChemTorch: A Deep Learning Framework for Benchmarking and Developing Chemical Reaction Property Prediction Models. J. Chem. Inf. Model. 2026, 66, 2434-2442, DOI: 10.1021/acs.jcim.5c02645